In [ ]:
"""
코랩용 기존 파이프라인 vs. 파인튜닝된 파이프라인 비교 스크립트
./adapters_dpo에 존재하는 어댑터를 기준으로 GPT-Score / 비GPT-Score를 비교.
random_persona_campaign.csv의 더미 데이터를 기준으로 평가함.
비교 문서는 adapter_comparison_{timestamp}.md로 저장.
"""

In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!pip install datasets peft trl bitsandbytes accelerate
!pip install -U transformers
!pip show transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.5/532.5 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 118.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
Name: transformers
Version: 4.57.6
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, r

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'drive', '.env', '.ipynb_checkpoints', 'sample_data']


In [5]:
!git clone https://github.com/jjjh02/AmoRe_crm_generator.git
%cd AmoRe_crm_generator
!git checkout jinhyeok
!git branch
os.chdir("/content/AmoRe_crm_generator/finetuning")
print(os.getcwd())

Cloning into 'AmoRe_crm_generator'...
remote: Enumerating objects: 438, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 438 (delta 90), reused 99 (delta 40), pack-reused 262 (from 1)
Receiving objects: 100% (438/438), 4.88 MiB | 3.46 MiB/s, done.
Resolving deltas: 100% (245/245), done.
/content/AmoRe_crm_generator
Branch 'jinhyeok' set up to track remote branch 'jinhyeok' from 'origin'.
Switched to a new branch 'jinhyeok'
* jinhyeok
  main
/content/AmoRe_crm_generator/finetuning


In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
import importlib
importlib.reload(sys.modules["run_qwen_exaone_pipeline"])

KeyError: 'run_qwen_exaone_pipeline'

In [7]:
#!/usr/bin/env python3
import argparse
import csv
import json
import os
import re
import sys
import urllib.error
import urllib.request
from collections import Counter
from contextlib import contextmanager
from datetime import datetime, timezone


BASE_DIR = os.getcwd()
PROJECT_DIR = os.path.abspath(os.path.join(BASE_DIR, ".."))
SRC_DIR = os.path.abspath(os.path.join(BASE_DIR, "..", "src"))

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from rag_utils import vectorize_texts, cosine
DEFAULT_CSV = os.path.join(BASE_DIR, "random_persona_campaign.csv")
DEFAULT_ADAPTER1_DIR = "/content/drive/MyDrive/LikeLion/adapters_kto_v1"
STAGE_ORDER = ["Acquisition", "Activation", "Retention", "Revenue", "Referral"]
CANDIDATE_LABELS = ["raw", "adapter1"]

print(BASE_DIR, SRC_DIR)

def _log(message):
    print(message)


def _import_pipeline_module():
    if SRC_DIR not in sys.path:
        sys.path.insert(0, SRC_DIR)
    try:
        import run_qwen_exaone_pipeline as pipeline_module
    except Exception as exc:
        raise ImportError(
            "Failed to import main from ../src/run_qwen_exaone_pipeline.py"
        ) from exc
    return pipeline_module


def _load_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return None


def _parse_bool(value):
    if isinstance(value, bool):
        return value
    if value is None:
        return False
    if isinstance(value, (int, float)):
        return bool(value)
    text = str(value).strip().lower()
    return text in {"1", "true", "yes", "y", "t"}


def _load_rows(csv_path):
    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row:
                continue
            persona_raw = row.get("persona", "").strip()
            brand_raw = row.get("brand", "").strip()
            product_raw = row.get("product", "").strip()
            stage_raw = row.get("stage_index", "").strip()
            style_raw = row.get("style_index", "").strip()
            if not persona_raw or not brand_raw or not product_raw:
                continue
            if not stage_raw or not style_raw:
                continue
            try:
                persona = int(persona_raw)
                stage_index = int(stage_raw)
                style_index = int(style_raw)
            except ValueError:
                continue
            yield {
                "persona": persona,
                "brand": brand_raw,
                "product": product_raw,
                "stage_index": stage_index,
                "style_index": style_index,
                "is_event": _parse_bool(row.get("is_event", "")),
            }


def _get_stage_name(stage_index):
    if isinstance(stage_index, int) and 0 <= stage_index < len(STAGE_ORDER):
        return STAGE_ORDER[stage_index]
    return ""


def _get_crm_goal(crm_goals, stage_index, stage_name=None):
    if not isinstance(crm_goals, dict):
        return {}
    if stage_name and stage_name in crm_goals:
        return crm_goals.get(stage_name, {}) or {}
    stage_name = _get_stage_name(stage_index)
    if stage_name:
        return crm_goals.get(stage_name, {}) or {}
    return {}


def _get_brand_story(brand_stories, brand_name):
    if not isinstance(brand_stories, dict) or not brand_name:
        return {}
    if brand_name in brand_stories:
        return brand_stories.get(brand_name, {}) or {}
    for story in brand_stories.values():
        if str(story.get("name_en", "")).lower() == brand_name.lower():
            return story
    return {}


def _format_event(selected_event):
    if selected_event in (None, "", {}):
        return "없음"
    if isinstance(selected_event, dict):
        for key in ("title", "name", "event_name", "event"):
            if selected_event.get(key):
                return str(selected_event.get(key))
        return json.dumps(selected_event, ensure_ascii=False)
    return str(selected_event)


def _format_price(price):
    if price in (None, ""):
        return ""
    if isinstance(price, (int, float)):
        return f"{int(price):,}원"
    text = str(price).strip()
    if not text:
        return ""
    if text.replace(",", "").isdigit():
        return f"{int(text.replace(',', '')):,}원"
    return text


def _format_persona(persona_profile):
    if not isinstance(persona_profile, dict):
        return str(persona_profile or "")
    name = persona_profile.get("name", "")
    extras = []
    value_focus = persona_profile.get("value_focus")
    skin_type = persona_profile.get("skin_type")
    traits = persona_profile.get("traits")
    shopping_style = persona_profile.get("shopping_style")
    if value_focus:
        extras.append(str(value_focus))
    if skin_type:
        extras.append(str(skin_type))
    if traits:
        if isinstance(traits, list):
            extras.append(", ".join([str(t) for t in traits if t]))
        else:
            extras.append(str(traits))
    if shopping_style:
        extras.append(str(shopping_style))
    extra_text = ", ".join([e for e in extras if e])
    if name and extra_text:
        return f"{name} ({extra_text})"
    return name or extra_text


def _build_context_block(out, max_style_templates=3):
    persona = _format_persona(out.get("persona_profile"))
    stage = out.get("stage_name") or out.get("stage_kr") or ""
    brand = out.get("brand") or ""
    product_basic = out.get("product_basic") if isinstance(out.get("product_basic"), dict) else {}
    product_name = product_basic.get("name") or out.get("product_query") or ""
    price = _format_price(product_basic.get("price"))
    objective = out.get("objective") or ""
    target_state = out.get("target_state") or ""
    style_templates = out.get("style_templates") or []
    if isinstance(style_templates, list):
        style_templates = style_templates[:max_style_templates]
    selected_event = _format_event(out.get("selected_event"))

    lines = ["[컨텍스트]"]
    if persona:
        lines.append(f"- 페르소나: {persona}")
    if stage:
        lines.append(f"- 단계: {stage}")
    if brand or product_name:
        if brand and product_name:
            brand_product = f"{brand} / {product_name}"
        else:
            brand_product = brand or product_name
        lines.append(f"- 브랜드/제품: {brand_product}")
    if price:
        lines.append(f"- 가격: {price}")
    if objective:
        lines.append(f"- 목표: {objective}")
    if target_state:
        lines.append(f"- 목표 상태: {target_state}")
    if style_templates:
        lines.append("- 스타일 템플릿:")
        for item in style_templates:
            lines.append(f"  - {item}")
    lines.append(f"- 이벤트: {selected_event}")
    return "\n".join(lines).strip()


def _extract_message(out):
    exaone = out.get("exaone", {}) if isinstance(out, dict) else {}
    return exaone.get("result_raw") or ""


def _tokenize(text):
    if not text:
        return []
    return [t for t in re.split(r"\s+", str(text)) if len(t) > 1]


def _split_tokens(text):
    if not text:
        return []
    cleaned = re.sub(r"[^\w\uac00-\ud7a3]+", " ", str(text), flags=re.UNICODE)
    return [t for t in cleaned.split() if len(t) > 1]


def _extract_keywords(texts, max_terms=30):
    counter = Counter()
    for text in texts:
        for token in _split_tokens(text):
            if token.isdigit():
                continue
            counter[token] += 1
    if not counter:
        return []
    return [item for item, _ in counter.most_common(max_terms)]


def _embedding_sim(message, ref_text):
    if not message or not ref_text:
        return 0.0
    vectors = vectorize_texts([message, ref_text])
    if vectors is None or len(vectors) < 2:
        return 0.0
    return float(cosine(vectors[0], vectors[1]))


def _persona_text(out):
    persona = out.get("persona_profile") if isinstance(out, dict) else {}
    if not isinstance(persona, dict):
        return str(persona or "")
    bits = []
    for key in ("name", "skin_type", "value_focus", "shopping_style", "growth_point"):
        val = persona.get(key)
        if val:
            bits.append(f"{key}: {val}")
    traits = persona.get("traits")
    if isinstance(traits, list) and traits:
        bits.append("traits: " + ", ".join([str(t) for t in traits if t]))
    return "
".join(bits)


def _brand_text(brand_story, brand_name):
    if not isinstance(brand_story, dict):
        brand_story = {}
    tone_keywords = brand_story.get("tone_keywords") or []
    story = brand_story.get("story") or ""
    parts = [brand_name or "", story]
    if tone_keywords:
        parts.append("tone_keywords: " + ", ".join([str(t) for t in tone_keywords if t]))
    return "
".join([p for p in parts if p])


def _purpose_text(out, crm_goal, stage_name):
    if not isinstance(crm_goal, dict):
        crm_goal = {}
    stage_kr = out.get("stage_kr") or crm_goal.get("stage_kr") or ""
    objective = crm_goal.get("objective") or out.get("objective") or ""
    target_state = crm_goal.get("target_state") or out.get("target_state") or ""
    cta_style = crm_goal.get("cta_style") or ""
    parts = [f"stage: {stage_name}", stage_kr, objective, target_state, cta_style]
    return "
".join([p for p in parts if p])


def _style_event_text(out):
    style_templates = out.get("style_templates") or []
    if not isinstance(style_templates, list):
        style_templates = [str(style_templates)]
    selected_event = _format_event(out.get("selected_event"))
    parts = []
    if style_templates:
        parts.append("style_templates:
" + "
".join([str(t) for t in style_templates if t]))
    if selected_event and selected_event != "??":
        parts.append("event: " + str(selected_event))
    return "
".join(parts)


def _product_text(out):
    brand = out.get("brand") or ""
    product_basic = out.get("product_basic") if isinstance(out.get("product_basic"), dict) else {}
    product_name = product_basic.get("name") or out.get("product_query") or ""
    price = _format_price(product_basic.get("price"))
    parts = [brand, product_name, price]
    return "
".join([p for p in parts if p])


def _clarity_reference():
    return "???? ??? CRM ???, ???? ???? ??? ??, ??? ??"


def _score_message(message, base_out, brand_story, crm_goal, stage_name):
    return {
        "len": len(message) if message else 0,
        "persona_sim": _embedding_sim(message, _persona_text(base_out)),
        "brand_sim": _embedding_sim(message, _brand_text(brand_story, base_out.get("brand"))),
        "purpose_sim": _embedding_sim(message, _purpose_text(base_out, crm_goal, stage_name)),
        "style_event_sim": _embedding_sim(message, _style_event_text(base_out)),
        "product_sim": _embedding_sim(message, _product_text(base_out)),
        "clarity_sim": _embedding_sim(message, _clarity_reference()),
    }


def _call_gpt(context_block, messages):
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set.")

    candidate_block = "\n\n".join(
        f"[{idx}]\n{msg if msg else '(빈 메시지)'}" for idx, msg in enumerate(messages)
    )

    system_prompt = (
    "?? CRM ??? ????.\n"
    "?? ???? ??? ????.\n"
    "1. persona? ??? CRM ????\n"
    "2. ??? ?? ? ?????\n"
    "3. ?? ??? ? ?????\n"
    "4. ??? ???/??? ??? ? ??????\n"
    "5. ??? ?? ??? ??? ?????\n"
    "6. ???? ???? ??? ??, ??? ?? ????\n"
    "?? ??? ??? 0 ?? 1? ????."
)

user_prompt = (
        "컨텍스트:\n"
        f"{context_block}\n\n"
        "후보:\n"
        f"{candidate_block}\n\n"
        "번호만 답해라."
    )

    payload = {
        "model": "gpt-4o-mini",
        "input": [
            {
                "role": "system",
                "content": [{"type": "input_text", "text": system_prompt}],
            },
            {
                "role": "user",
                "content": [{"type": "input_text", "text": user_prompt}],
            },
        ],
    }

    request = urllib.request.Request(
        "https://api.openai.com/v1/responses",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"OpenAI API error {exc.code}: {body}") from exc

    output_text = _extract_response_text(data)
    match = re.search(r"-?\d+", str(output_text))
    if not match:
        raise ValueError(f"Invalid evaluator response: {output_text}")
    choice = int(match.group(0))
    if choice not in (0, 1):
        raise ValueError(f"Evaluator index out of range: {choice}")
    return choice


def _extract_response_text(data):
    if isinstance(data, dict):
        output_text = data.get("output_text")
        if isinstance(output_text, str) and output_text.strip():
            return output_text.strip()

        output = data.get("output")
        if isinstance(output, list):
            parts = []
            for item in output:
                if not isinstance(item, dict):
                    continue
                content = item.get("content", [])
                if isinstance(content, list):
                    for block in content:
                        if isinstance(block, dict) and isinstance(block.get("text"), str):
                            parts.append(block["text"])
                        elif isinstance(block, str):
                            parts.append(block)
                elif isinstance(content, str):
                    parts.append(content)
            if parts:
                return "".join(parts).strip()

    return ""


@contextmanager
def _patch_exaone(pipeline_module, adapter_path=None, adapter_paths=None):
    import tone_correction

    class PatchedExaoneToneCorrector(tone_correction.ExaoneToneCorrector):
        _cache = {}

        def __init__(self, model_name="LGAI-EXAONE/EXAONE-4.0-1.2B"):
            if adapter_paths:
                key = (model_name, tuple(adapter_paths))
            else:
                key = (model_name, adapter_path)
            cached = self._cache.get(key)
            if cached:
                self.device = cached["device"]
                self.model_name = model_name
                self.tokenizer = cached["tokenizer"]
                self.model = cached["model"]
                return
            super().__init__(model_name=model_name)
            if adapter_paths:
                self._apply_adapters(adapter_paths)
            elif adapter_path:
                self._apply_adapters([adapter_path])
            self._cache[key] = {
                "device": self.device,
                "tokenizer": self.tokenizer,
                "model": self.model,
            }

        def _apply_adapters(self, paths):
            if not paths:
                return
            try:
                from peft import PeftModel
            except ImportError as exc:
                raise RuntimeError("peft is required to load adapters.") from exc

            self.model = PeftModel.from_pretrained(self.model, paths[0])
            if len(paths) == 1:
                try:
                    self.model.eval()
                except Exception:
                    pass
                return

            merged = None
            try:
                merged = self.model.merge_and_unload()
            except Exception:
                merged = None

            if merged is not None:
                self.model = merged
                self.model = PeftModel.from_pretrained(self.model, paths[1])
            else:
                try:
                    self.model.load_adapter(paths[1], adapter_name="adapter2")
                    try:
                        self.model.set_adapter(["default", "adapter2"])
                    except Exception:
                        self.model.set_adapter("adapter2")
                except Exception:
                    pass

            try:
                self.model.eval()
            except Exception:
                pass

    original = pipeline_module.ExaoneToneCorrector
    pipeline_module.ExaoneToneCorrector = PatchedExaoneToneCorrector
    try:
        yield
    finally:
        pipeline_module.ExaoneToneCorrector = original


def _run_pipeline_main(pipeline_main, row):
    argv = [
        "run_qwen_exaone_pipeline.py",
        "--persona",
        str(row["persona"]),
        "--brand",
        row["brand"],
        "--product",
        row["product"],
        "--stage_index",
        str(row["stage_index"]),
        "--style_index",
        str(row["style_index"]),
        "--is_event",
        "1" if row.get("is_event", False) else "0",
    ]
    old_argv = sys.argv
    try:
        sys.argv = argv
        return pipeline_main()
    finally:
        sys.argv = old_argv


def _row_key(row):
    return "{persona}|{brand}|{product}|{stage}|{style}|{event}".format(
        persona=row.get("persona", ""),
        brand=row.get("brand", ""),
        product=row.get("product", ""),
        stage=row.get("stage_index", ""),
        style=row.get("style_index", ""),
        event=int(bool(row.get("is_event", False))),
    )


def _load_checkpoint(checkpoint_path, row_key_map):
    if not checkpoint_path or not os.path.exists(checkpoint_path):
        return [], set()
    results = []
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            row_key = item.get("row_key")
            if row_key and row_key in row_key_map:
                item["idx"] = row_key_map[row_key]
            if "idx" not in item:
                continue
            results.append(item)
    by_idx = {}
    for item in results:
        by_idx[item["idx"]] = item
    ordered = [by_idx[idx] for idx in sorted(by_idx)]
    return ordered, set(by_idx)


def _append_checkpoint(checkpoint_path, item):
    if not checkpoint_path:
        return
    checkpoint_dir = os.path.dirname(checkpoint_path)
    if checkpoint_dir:
        os.makedirs(checkpoint_dir, exist_ok=True)
    with open(checkpoint_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")


def _count_wins(results):
    wins = {"raw": 0, "adapter1": 0}
    mapping = {
        "raw": "raw",
        "ad1": "adapter1",
        "adapter1": "adapter1",
    }
    for item in results:
        key = mapping.get(item.get("winner"))
        if key:
            wins[key] += 1
    return wins


def _write_report(out_path, summary, rows, max_examples):
    lines = []
    lines.append("# 어댑터 비교 리포트")
    lines.append("")
    lines.append(f"- CSV: {summary['csv']}")
    lines.append(f"- 어댑터1: {summary['adapter1']}")
    lines.append(f"- 샘플 수: {summary['samples']}")
    lines.append("- 표기: raw=기본 모델, ad1=어댑터1")
    lines.append("")
    lines.append("## 요약")
    lines.append("")
    for item in summary["metrics"]:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## 지표 설명")
    lines.append("")
    lines.append("- GPT 승자: gpt-4o-mini가 동일 컨텍스트 기준으로 2개 후보 중 더 좋은 메시지를 선택한 결과.")
    lines.append("- 커버리지: 브랜드/제품/이벤트/스테이지 관련 용어가 메시지에 포함된 비율.")
    lines.append("- 톤 일치율: 브랜드 톤 키워드가 메시지에 포함된 비율.")
    lines.append("- 스타일 일치율: 스타일 템플릿에서 뽑은 키워드 포함 비율.")
    lines.append("- 정보 밀도: 컨텍스트 키워드 적중 수 / 메시지 토큰 수.")
    lines.append("- 반복 토큰 비율: (토큰 수 - 고유 토큰 수) / 토큰 수.")
    lines.append("- 반복 3-그램 비율: 반복된 3-그램 수 / 전체 3-그램 수.")
    lines.append("- 길이 적정: 스테이지별 권장 길이 범위 충족 여부.")
    lines.append("- 금지 맥락 위반율: forbidden_context 용어가 포함된 메시지 비율.")
    lines.append("- CTA 비율: CTA 키워드가 포함된 메시지 비율.")
    lines.append("")
    lines.append("## 샘플별 결과")
    lines.append("")
    lines.append(
        "| idx | persona | 브랜드/제품 | 스테이지 | 이벤트 | GPT 승자 | raw 길이 | ad1 길이 | raw 커버리지 | ad1 커버리지 | raw 톤 | ad1 톤 | raw 스타일 | ad1 스타일 | raw 밀도 | ad1 밀도 | raw 반복 토큰 | ad1 반복 토큰 | raw 반복 3g | ad1 반복 3g | raw 길이 적정 | ad1 길이 적정 | raw 금지 | ad1 금지 | raw CTA | ad1 CTA |"
    )
    lines.append(
        "| --- | --- | --- | --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |"
    )
    for item in rows:
        lines.append(
            "| {idx} | {persona} | {brand_product} | {stage} | {event} | {winner} | {raw_len} | {ad1_len} | {raw_persona:.2f} | {ad1_persona:.2f} | {raw_brand:.2f} | {ad1_brand:.2f} | {raw_purpose:.2f} | {ad1_purpose:.2f} | {raw_purpose_event:.2f} | {ad1_purpose_event:.2f} | {raw_product:.2f} | {ad1_product:.2f} | {raw_clarity:.2f} | {ad1_clarity:.2f} |".format(
                idx=item["idx"],
                persona=item["persona"],
                brand_product=item["brand_product"],
                stage=item["stage"],
                event=item["event"],
                winner=item["winner"],
                raw_len=item["raw_len"],
                ad1_len=item["adapter1_len"],
                raw_persona=item["raw_persona"],
                ad1_persona=item["adapter1_cov"],
                raw_brand=item["raw_brand"],
                ad1_brand=item["adapter1_tone"],
                raw_purpose=item["raw_purpose"],
                ad1_purpose=item["adapter1_style"],
                raw_style_event=item["raw_style_event"],
                ad1_style_event=item["adapter1_density"],
                raw_product=item["raw_product"],
                ad1_product=item["adapter1_rep_token"],
                raw_clarity=item["raw_clarity"],
                ad1_clarity=item["adapter1_rep_ngram"],
                ="yes" if item[""] else "no",
                ="yes" if item["adapter1_len_ok"] else "no",
                =item[""],
                =item["adapter1_forbidden"],
                ="yes" if item[""] else "no",
                ="yes" if item["adapter1_cta"] else "no",
            )
        )
    lines.append("")

    example_count = min(max_examples, len(rows))
    if example_count > 0:
        lines.append("## 예시")
        lines.append("")
        for item in rows[:example_count]:
            lines.append(f"### 샘플 {item['idx']}")
            lines.append("")
            lines.append("컨텍스트:")
            lines.append("")
            lines.append("```")
            lines.append(item["context"])
            lines.append("```")
            lines.append("")
            lines.append("Raw 메시지:")
            lines.append("")
            lines.append("```")
            lines.append(item["raw_message"] or "(빈 메시지)")
            lines.append("```")
            lines.append("")
            lines.append("Adapter1 메시지:")
            lines.append("")
            lines.append("```")
            lines.append(item["adapter1_message"] or "(빈 메시지)")
            lines.append("```")
            lines.append("")
            lines.append(f"GPT 승자: {item['winner']}")
            lines.append("")
    else:
        lines.append("## 예시")
        lines.append("")
        lines.append("예시가 없습니다 (max_examples가 0이거나 처리된 행이 없습니다).")
        lines.append("")

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))




/content/AmoRe_crm_generator/finetuning /content/AmoRe_crm_generator/src


In [16]:
parser = argparse.ArgumentParser()
parser.add_argument("--csv_path", default=DEFAULT_CSV)
parser.add_argument("--adapter1_path", default=DEFAULT_ADAPTER1_DIR)
parser.add_argument("--out_path", default=None)                       # 아래에서 별도 지정
parser.add_argument("--checkpoint_path", default=None)                # /content 하위
parser.add_argument("--max_rows", type=int, default=10)             # 분석 샘플 수
parser.add_argument("--max_examples", type=int, default=10)
parser.add_argument("--skip_llm_eval", action="store_true")
parser.add_argument("--max_style_templates", type=int, default=3)
args, _ = parser.parse_known_args()

if not os.path.exists(args.csv_path):
    raise FileNotFoundError(f"CSV not found: {args.csv_path}")
if not os.path.exists(args.adapter1_path):
    raise FileNotFoundError(f"Adapter not found: {args.adapter1_path}")

pipeline_module = _import_pipeline_module()
pipeline_main = pipeline_module.main
brand_stories = _load_json(os.path.join(PROJECT_DIR, "data", "brand_stories.json"))
crm_goals = _load_json(os.path.join(PROJECT_DIR, "data", "crm_goals.json"))

rows = []
row_key_map = {}
for idx, row in enumerate(_load_rows(args.csv_path), start=1):
    if args.max_rows is not None and idx > args.max_rows:
        break
    rows.append(row)
    key = _row_key(row)
    if key in row_key_map:
        _log(f"[WARN] Duplicate row key at idx={idx}.")
    else:
        row_key_map[key] = idx

if not rows:
    raise RuntimeError("No rows to evaluate.")

checkpoint_path = args.checkpoint_path or os.path.join(
    BASE_DIR, "adapter_comparison_2way_checkpoint.jsonl"
)
results, processed = _load_checkpoint(checkpoint_path, row_key_map)
if results:
    _log(f"Resume from checkpoint: {checkpoint_path} ({len(results)} rows)")

wins = _count_wins(results)

for idx, row in enumerate(rows, start=1):
    if idx in processed:
        _log(f"[Row {idx}] skipped (checkpoint)")
        continue
    _log(
        "[Row {idx}] persona={persona} brand={brand} product={product} "
        "stage_index={stage_index} style_index={style_index} is_event={is_event}".format(
            idx=idx,
            persona=row["persona"],
            brand=row["brand"],
            product=row["product"],
            stage_index=row["stage_index"],
            style_index=row["style_index"],
            is_event=row.get("is_event", False),
        )
    )

    _log("  Running raw pipeline...")
    raw_out = _run_pipeline_main(pipeline_main, row)

    _log("  Running adapter1 pipeline...")
    with _patch_exaone(pipeline_module, adapter_path=args.adapter1_path):
        adapter1_out = _run_pipeline_main(pipeline_main, row)

    raw_message = _extract_message(raw_out)
    adapter1_message = _extract_message(adapter1_out)

    context_block = _build_context_block(raw_out, args.max_style_templates)
    stage_name = raw_out.get("stage_name") or _get_stage_name(row["stage_index"])
    crm_goal = _get_crm_goal(crm_goals, row["stage_index"], stage_name)
    brand_story = _get_brand_story(brand_stories, raw_out.get("brand"))

    winner = "n/a"
    if not args.skip_llm_eval:
        choice = _call_gpt(context_block, [raw_message, adapter1_message])
        winner = CANDIDATE_LABELS[choice]
        wins[winner] += 1

    raw_metrics = _score_message(raw_message, raw_out, brand_story, crm_goal, stage_name)
    adapter1_metrics = _score_message(adapter1_message, raw_out, brand_story, crm_goal, stage_name)

    winner_short = {"raw": "raw", "adapter1": "ad1"}.get(winner, "n/a")
    row_key = _row_key(row)

    result = {
        "idx": idx,
        "row_key": row_key,
        "persona": row["persona"],
        "brand_product": f"{row['brand']} / {row['product']}",
        "stage": raw_out.get("stage_name") or raw_out.get("stage_kr") or "",
        "event": _format_event(raw_out.get("selected_event")),
        "winner": winner_short,
        "raw_len": raw_metrics["len"],
        "adapter1_len": adapter1_metrics["len"],
        "raw_persona": raw_metrics["persona_sim"],
        "adapter1_persona": adapter1_metrics["persona_sim"],
        "raw_brand": raw_metrics["brand_sim"],
        "adapter1_brand": adapter1_metrics["brand_sim"],
        "raw_purpose": raw_metrics["purpose_sim"],
        "adapter1_purpose": adapter1_metrics["purpose_sim"],
        "raw_style_event": raw_metrics["style_event_sim"],
        "adapter1_style_event": adapter1_metrics["style_event_sim"],
        "raw_product": raw_metrics["product_sim"],
        "adapter1_product": adapter1_metrics["product_sim"],
        "raw_clarity": raw_metrics["clarity_sim"],
        "adapter1_clarity": adapter1_metrics["clarity_sim"],
        "context": context_block,
        "raw_message": raw_message,
        "adapter1_message": adapter1_message,
    }
    results.append(result)
    _append_checkpoint(checkpoint_path, result)


def _avg_metric(results, key):
    values = [r[key] for r in results if key in r]
    return sum(values) / len(values) if values else 0.0


results = sorted(results, key=lambda item: item.get("idx", 0))
total = len(results) if results else 1

avg_persona = {c: _avg_metric(results, f"{c}_persona") for c in ("raw", "adapter1")}
avg_brand = {c: _avg_metric(results, f"{c}_brand") for c in ("raw", "adapter1")}
avg_purpose = {c: _avg_metric(results, f"{c}_purpose") for c in ("raw", "adapter1")}
avg_style_event = {c: _avg_metric(results, f"{c}_style_event") for c in ("raw", "adapter1")}
avg_product = {c: _avg_metric(results, f"{c}_product") for c in ("raw", "adapter1")}
avg_clarity = {c: _avg_metric(results, f"{c}_clarity") for c in ("raw", "adapter1")}
avg_len = {c: _avg_metric(results, f"{c}_len") for c in ("raw", "adapter1")}

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out_path = args.out_path or os.path.join(
    "/content/drive/MyDrive/LikeLion/comparison_report", f"adapter_kto_v1_{timestamp}.md"
)

summary = {
    "csv": args.csv_path,
    "adapter1": args.adapter1_path,
    "samples": len(results),
    "metrics": [
        "GPT ??: raw {raw} / ad1 {ad1} (skip_llm_eval={skip})".format(
            raw=wins["raw"],
            ad1=wins["adapter1"],
            skip=args.skip_llm_eval,
        ),
        "?? persona ???: raw {raw:.2f}, ad1 {ad1:.2f}".format(
            raw=avg_persona["raw"], ad1=avg_persona["adapter1"]
        ),
        "?? ??? ? ???: raw {raw:.2f}, ad1 {ad1:.2f}".format(
            raw=avg_brand["raw"], ad1=avg_brand["adapter1"]
        ),
        "?? ?? ?? ???: raw {raw:.2f}, ad1 {ad1:.2f}".format(
            raw=avg_purpose["raw"], ad1=avg_purpose["adapter1"]
        ),
        "?? ???/??? ???: raw {raw:.2f}, ad1 {ad1:.2f}".format(
            raw=avg_style_event["raw"], ad1=avg_style_event["adapter1"]
        ),
        "?? ?? ???: raw {raw:.2f}, ad1 {ad1:.2f}".format(
            raw=avg_product["raw"], ad1=avg_product["adapter1"]
        ),
        "?? ??? ???: raw {raw:.2f}, ad1 {ad1:.2f}".format(
            raw=avg_clarity["raw"], ad1=avg_clarity["adapter1"]
        ),
        "?? ??: raw {raw:.1f}, ad1 {ad1:.1f}".format(
            raw=avg_len["raw"], ad1=avg_len["adapter1"]
        ),
    ],
}

_write_report(out_path, summary, results, args.max_examples)
_log(f"Saved report: {out_path}")



[Row 1] persona=2 brand=에뛰드 product=뽀오얀 미소 발효 립&아이 리무버 250ml (대용량) stage_index=4 style_index=0 is_event=False
  Running raw pipeline...
Loading SentenceTransformer model (jhgan/ko-sroberta-multitask)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

[Exaone] 모델 로딩 완료


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

Log saved: /content/AmoRe_crm_generator/log/pipeline_에뛰드_Referral_20260118T075043Z.json
Result saved: /content/AmoRe_crm_generator/outputs/result_에뛰드_20260118T075043Z.json
--- Qwen draft ---
### [뽀오얀 미소 발효 립&아이 리무버 250ml]

#### [제목]
 efficacious & affordable lip and eye remover for all your needs

#### [본문]
As a long-time user of the EDDY lip and eye remover, I've noticed that despite its slightly higher price point compared to other large-volume options on the market, it consistently delivers excellent results without sacrificing value. The product's effectiveness is undeniable; it leaves my eyes feeling refreshed and well-cleansed every time I use it. 

The unique formulation ensures that even after multiple uses, my eyes remain clear and free from any residue. Unlike some competitors, this product doesn't leave an unpleasant haze in my vision, making it a refreshing choice for those who prefer not to see anything but their makeup. Its affordability makes it accessible to everyone, r